# 02 - Feature Engineering & Preprocessing

This notebook demonstrates how raw financial data is cleaned, transformed into domain-specific financial ratios, and split without customer leakage:
1. Cleaning corrupted numeric strings (`_`, `-500`, `__10000__`, `__-333...__`) and categorical placeholders (`_______`, `!@9#%8`).
2. Parsing `Credit_History_Age` into `Credit_History_Months`.
3. Engineering financial health ratios (`Debt_to_Income_Ratio`, `EMI_to_Salary_Ratio`, `Savings_to_Salary_Ratio`, `Delayed_Payment_Rate`, `Delay_Severity_Score`).
4. Group-aware `Customer_ID` train/test splitting and `ColumnTransformer` preprocessing.


In [ ]:
import os
import sys

sys.path.append(os.path.abspath(".."))
from src.data_loader import load_raw_data
from src.preprocessing import clean_credit_dataframe, split_dataset, build_preprocessor
from src.feature_engineering import engineer_features, NUMERIC_FEATURES, CATEGORICAL_FEATURES

df = load_raw_data(os.path.abspath("../dataset/raw/train.csv"))


## 1. Group-Aware Split & Cleaning


In [ ]:
X_train_raw, X_test_raw, y_train, y_test = split_dataset(df, test_size=0.2, random_state=42)
print(f"Train rows: {len(X_train_raw)}, Test rows: {len(X_test_raw)}")

X_train_clean = clean_credit_dataframe(X_train_raw)
X_train_eng = engineer_features(X_train_clean)
X_train_eng[["Debt_to_Income_Ratio", "EMI_to_Salary_Ratio", "Savings_to_Salary_Ratio", "Delayed_Payment_Rate", "Credit_History_Months"]].describe().T


## 2. Scikit-Learn ColumnTransformer Pipeline


In [ ]:
preprocessor = build_preprocessor(NUMERIC_FEATURES, CATEGORICAL_FEATURES)
X_train_proc = preprocessor.fit_transform(X_train_eng)
print("Transformed feature matrix shape:", X_train_proc.shape)
print("Total output features:", len(preprocessor.get_feature_names_out()))
